Exercise 1: Create imports, DAG argument, and definition
Please use the BashOperator for all tasks in this assignment.
Create a new file named ETL_toll_data.py in /home/project directory and open it in the file editor.

Import all the packages you need to build the DAG.

Task 1.1: Define DAG arguments
Define the DAG arguments as per the following details in the ETL_toll_data.py file:

| Parameter | Value | | ----------------- | ------- | | owner | | | start_date | today | | email | | | email_on_failure | True | | email_on_retry | True| | retries | 1| | retry_delay | 5 minutes|

Take a screenshot of the task code. Name the screenshot dag_args.jpg.

Task 1.2: Define the DAG
Define the DAG in the ETL_toll_data.py file using the following details.

| Parameter | Value | | ----------------- | ------- | | DAG id | ETL_toll_data | | Schedule | Daily once | | default_args | As you have defined in the previous step | | description | Apache Airflow Final Assignment |

Take a screenshot of the command and output you used. Name the screenshot dag_definition.jpg.

At the end of this exercise, you should have the following screenshots with .jpg or .png extension: 1. dag_args.jpg 2. dag_definition.jpg

In [ ]:
# 1. To Handle Date and time
from datetime import timedelta
from airflow import DAG

# 2. Because I will use Bash command so I am using BashOperator.
from airflow.operators.bash import BashOperator

# 3. Date format assign start_date
from airflow.utils.dates import days_ago

# Define the args 
default_args = {
    'owner': 'Joy Sen', 
    'start_date': days_ago(0),
    'email': ['santunu23@gmail.com'],
    'email_on_failure': True,        
    'email_on_retry': True,          
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

dag = DAG(
    'ETL_toll_data',
    default_args=default_args,
    description='Apache Airflow Final Assignment',
    schedule_interval=timedelta(days=1),
)

Exercise 2: Create the tasks using BashOperator
Task 2.1: Create a task to unzip data.
Create a task named unzip_data to unzip data. Use the data downloaded in the first part of this assignment in Set up the lab environment and uncompress it into the destination directory using tar.

Take a screenshot of the task code. Name the screenshot unzip_data.jpg.

You can locally untar and read through the file fileformats.txt to understand the column details.

Task 2.2: Create a task to extract data from csv file
Create a task named extract_data_from_csv to extract the fields Rowid, Timestamp, Anonymized Vehicle number, and Vehicle type from the vehicle-data.csv file and save them into a file named csv_data.csv.

Take a screenshot of the task code. Name the screenshot extract_data_from_csv.jpg.

Task 2.3: Create a task to extract data from tsv file
Create a task named extract_data_from_tsv to extract the fields Number of axles, Tollplaza id, and Tollplaza code from the tollplaza-data.tsv file and save it into a file named tsv_data.csv.

Take a screenshot of the task code. Name the screenshot extract_data_from_tsv.jpg.

Task 2.4: Create a task to extract data from fixed width file
Create a task named extract_data_from_fixed_width to extract the fields Type of Payment code, and Vehicle Code from the fixed width file payment-data.txt and save it into a file named fixed_width_data.csv.

Take a screenshot of the task code. Name the screenshot extract_data_from_fixed_width.jpg.

Task 2.5: Create a task to consolidate data extracted from previous tasks
Create a task named consolidate_data to consolidate data extracted from previous tasks. This task should create a single csv file named extracted_data.csv by combining data from the following files:

- csv_data.csv - tsv_data.csv - fixed_width_data.csv

The final csv file should use the fields in the order given below:

- Rowid - Timestamp - Anonymized Vehicle number - Vehicle type - Number of axles - Tollplaza id - Tollplaza code - Type of Payment code, and - Vehicle Code

Hint: Use the bash paste command that merges the columns of the files passed as a command-line parameter and sends the output to a new file specified. You can use the command man paste to explore more.

Example: paste file1 file2 > newfile

Take a screenshot of the task code. Name the screenshot consolidate_data.jpg.

Task 2.6: Transform the data
Create a task named transform_data to transform the vehicle_type field in extracted_data.csv into capital letters and save it into a file named transformed_data.csv in the staging directory.

Hint: You can use the tr command within the BashOperator in Airflow.

Take a screenshot of the task code. Name the screenshot transform.jpg.

Task 2.7: Define the task pipeline
Define the task pipeline as per the details given below:

| Task | Functionality | | ----------------- | ------- | |First task | unzip_data | |Second task | extract_data_from_csv | |Third task | extract_data_from_tsv | |Fourth task | extract_data_from_fixed_width | |Fifth task | consolidate_data | |Sixth task | transform_data |

Take a screenshot of the task pipeline section of the DAG. Name the screenshot task_pipeline.jpg.

In [ ]:
# Task 2.1: Create a task to unzip data
unzip_data = BashOperator(
    task_id='unzip_data',
    bash_command='tar -xzf /home/project/airflow/dags/finalassignment/tolldata.tgz -C /home/project/airflow/dags/finalassignment',
    dag=dag,
)

# Task 2.2: Create a task to extract data from csv file
extract_data_from_csv = BashOperator(
    task_id='extract_data_from_csv',
    bash_command='cut -d"," -f1,2,3,4 /home/project/airflow/dags/finalassignment/vehicle-data.csv > /home/project/airflow/dags/finalassignment/csv_data.csv',
    dag=dag,
)

# Task 2.3: Create a task to extract data from tsv file
extract_data_from_tsv = BashOperator(
    task_id='extract_data_from_tsv',
    bash_command='cut -f5,6,7 /home/project/airflow/dags/finalassignment/tollplaza-data.tsv > /home/project/airflow/dags/finalassignment/tsv_data.csv',
    dag=dag,
)

# Task 2.4: Create a task to extract data from fixed width file
extract_data_from_fixed_width = BashOperator(
    task_id='extract_data_from_fixed_width',
    bash_command='cut -c59-62,63-67 /home/project/airflow/dags/finalassignment/payment-data.txt > /home/project/airflow/dags/finalassignment/fixed_width_data.csv',
    dag=dag,
)

# Task 2.5: Create a task to consolidate data extracted from previous tasks
consolidate_data = BashOperator(
    task_id='consolidate_data',
    bash_command='paste -d"," /home/project/airflow/dags/finalassignment/csv_data.csv /home/project/airflow/dags/finalassignment/tsv_data.csv /home/project/airflow/dags/finalassignment/fixed_width_data.csv > /home/project/airflow/dags/finalassignment/extracted_data.csv',
    dag=dag,
)

# Task 2.6: Transform the data
transform_data = BashOperator(
    task_id='transform_data',
    bash_command='awk -F"," \'BEGIN {OFS=","} {$4=toupper($4); print}\' /home/project/airflow/dags/finalassignment/extracted_data.csv > /home/project/airflow/dags/finalassignment/transformed_data.csv',
    dag=dag,
)

# Task 2.7: Define the task pipeline
unzip_data >> extract_data_from_csv >> extract_data_from_tsv >> extract_data_from_fixed_width >> consolidate_data >> transform_data

Exercise 3: Getting the DAG operational
Task 3.1: Submit the DAG
Submit the DAG. Use CLI or Web UI to show that the DAG has been properly submitted. Take a screenshot showing that the DAG you created is in the list of DAGs. Name the screenshot submit_dag.jpg.

Note: If you don't find your DAG in the list, you can check for errors using the following command in the terminal:


bash
airflow dags list-import-errors
Run
Task 3.2: Unpause and trigger the DAG
Unpause and trigger the DAG through CLI or Web UI.

Take a screenshot of DAG unpaused on CLI or the GUI. Name the screenshot unpause_trigger_dag.jpg.

Task 3.3: List the DAG tasks
Take a screenshot of the tasks in the DAG run through CLI or Web UI. Name the screenshot dag_tasks.jpg.

Task 3.4: Monitor the DAG
Take a screenshot the DAG runs for the Airflow console through CLI or Web UI. Name the screenshot dag_runs.jpg.